# Confronto sistematico degli ansatz per il trimero a catena aperta — con DM

Fase 5: sweep sistematico unico (RBS e $W$, $D=0$ e DM acceso), mirror di
`confronto_ansatz_entangler_trimero_anello.ipynb` ma con una differenza di
struttura dichiarata nel piano operativo — qui non servono due tappe separate
(indagine mirata poi confronto sistematico): l'ipotesi da testare (RBS ha un
tetto strutturale sotto DM, $W$ no) è già nota dall'anello, verificata qui sui
dati della catena, non assunta.

Punto di lavoro DM confermato in `analisi_dm_trimero_catena.pdf`: $J=1$,
$b_c=3.0$, $D=0.15$.

Ansatz e sweep importati da `ansatz_catena.py` / `sweep_catena_dm.py`
(libreria già verificata, non riscritta qui) — questo notebook ne è la vista
interattiva: circuiti, tabelle, grafici, e un ricalcolo live opzionale.

## 0. Setup

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from qiskit.quantum_info import Statevector

from trimer_chain_exact import critical_field, trimer_hamiltonian, trimer_hamiltonian_dm
from ansatz_catena import build_ansatze, pma_2q_trimer_cyclic, w_block, rbs_block

J = 1.0
D_DM = 0.15
BC = critical_field(J)
print(f"b_c = {BC},  D_DM = {D_DM}")

ANSATZE = build_ansatze()
# le due varianti a 2 giri (K2), aggiunte dopo la scoperta del tetto a 1 giro
ANSATZE["RBS-2qC.K2"] = pma_2q_trimer_cyclic(2, rbs_block)
ANSATZE["W-2qC.K2"] = pma_2q_trimer_cyclic(2, w_block)

for name, qc in ANSATZE.items():
    print(f"  {name:14s} {qc.num_parameters:2d} parametri")

## 1. I circuiti — selettore interattivo

Stesso pattern di `analisi_espressivita_PMA_anello.ipynb`: vista statica di
default sempre visibile riaprendo il file, selettore live se il kernel è
acceso.

In [ ]:
def mostra_circuito(nome):
    qc = ANSATZE[nome]
    ops = qc.decompose().count_ops()
    n_2q = ops.get("cz", 0) + ops.get("cx", 0)
    n_tot = sum(ops.values())
    print(f"{nome} -- {qc.num_parameters} parametri, {n_2q} gate a due qubit, "
          f"{n_tot} gate totali")
    try:
        from IPython.display import display
        display(qc.draw("mpl", style="iqp"))
    except Exception:
        print(qc.draw())


print("Circuito di default (candidato canonico per la catena) -- W-2qC.K2:")
mostra_circuito("W-2qC.K2")

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

if HAS_WIDGETS:
    dropdown = widgets.Dropdown(
        options=list(ANSATZE.keys()), value="W-2qC.K2",
        description="Ansatz:", style={"description_width": "initial"},
    )
    out = widgets.Output()

    def _aggiorna(*_):
        with out:
            clear_output(wait=True)
            mostra_circuito(dropdown.value)

    dropdown.observe(_aggiorna, names="value")
    print("\nSelettore interattivo (richiede kernel Jupyter vivo):")
    display(dropdown, out)
    _aggiorna()
else:
    print("\nipywidgets non disponibile: nessun selettore live.")

## 2. Sweep — cache già calcolata (300/300 punti)

Cella di calcolo, riutilizzabile: se la cache è già completa non ricalcola
nulla (verificato: "0 punti nuovi"); se mancano punti (es. griglia estesa),
li calcola e salva incrementalmente.

In [ ]:
import time
from scipy.optimize import minimize
from qiskit.primitives import StatevectorEstimator
from trimer_chain_exact import exact_sweep, exact_sweep_dm, ground_state_projector, ground_state_projector_dm

B_GRID = sorted(set(np.linspace(0.05, 2 * BC, 9).round(3).tolist() + [BC]))
CACHE_PATH = "trimer_chain_dm_sweep_cache.json"
ESTIMATOR = StatevectorEstimator()


def _energy(params, ansatz, hamiltonian):
    pub = (ansatz, hamiltonian, [params])
    return float(ESTIMATOR.run([pub]).result()[0].data.evs.item())


def run_vqe(ansatz, b, D, n_restarts=2, seed=42):
    rng = np.random.default_rng(seed)
    if D != 0.0:
        hamiltonian = trimer_hamiltonian_dm(J, b, D)
        e_exact = float(exact_sweep_dm(np.array([b]), J, D)["gs_energy"][0])
        P0, _, deg = ground_state_projector_dm(J, b, D)
    else:
        hamiltonian = trimer_hamiltonian(J, b)
        e_exact = float(exact_sweep(np.array([b]), J=J)["gs_energy"][0])
        P0, _, deg = ground_state_projector(J, b)
    best_e, best_params = np.inf, None
    for _ in range(n_restarts):
        x0 = rng.uniform(-np.pi, np.pi, ansatz.num_parameters)
        r1 = minimize(_energy, x0, args=(ansatz, hamiltonian),
                      method="COBYLA", options={"maxiter": 2000, "tol": 1e-10})
        r2 = minimize(_energy, r1.x, args=(ansatz, hamiltonian),
                      method="L-BFGS-B", options={"maxiter": 2000, "ftol": 1e-14})
        if r2.fun < best_e:
            best_e, best_params = r2.fun, r2.x
    sv = Statevector(ansatz.assign_parameters(best_params)).data
    fid = float(np.real(sv.conj() @ P0 @ sv))
    return {"e_vqe": best_e, "e_exact": e_exact, "delta_e": best_e - e_exact,
            "fidelity": fid, "n_params": int(ansatz.num_parameters), "degeneracy": int(deg)}


def load_cache():
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH) as f:
            return json.load(f)
    return {}


def save_cache(cache):
    with open(CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=1)


cache = load_cache()
n_attesi = len(ANSATZE) * len(B_GRID) * 2
print(f"Punti attesi: {n_attesi}  |  già in cache: {len(cache)}")

t0 = time.time()
n_new = 0
for name, ansatz in ANSATZE.items():
    for b in B_GRID:
        for tag, D in [("D0", 0.0), ("DM", D_DM)]:
            key = f"{tag}|{name}|{b}"
            if key in cache:
                continue
            cache[key] = run_vqe(ansatz, b, D)
            n_new += 1
            save_cache(cache)
print(f"{n_new} punti nuovi in {time.time()-t0:.0f}s  ({len(cache)} totali in cache)")

## 3. Tabelle riassuntive

In [ ]:
print(f"{'ansatz':14s} {'par':>4s} | {'D0 min':>8s} {'D0 media':>9s} | {'DM min':>8s} {'DM media':>9s}")
print("-" * 70)
for name in ANSATZE:
    d0 = [v["fidelity"] for k, v in cache.items() if k.startswith(f"D0|{name}|")]
    dm = [v["fidelity"] for k, v in cache.items() if k.startswith(f"DM|{name}|")]
    if not d0:
        continue
    npar = ANSATZE[name].num_parameters
    print(f"{name:14s} {npar:4d} | {min(d0):8.4f} {sum(d0)/len(d0):9.4f} | "
          f"{min(dm):8.4f} {sum(dm)/len(dm):9.4f}")

## 4. Grafici — fidelity vs $b$, RBS vs $W$, $D=0$ vs DM

Famiglie a bassa espressività (`RBS-1q.3/6`, `RBS-2q.2`) escluse dai grafici:
limite strutturale già noto (non raggiungono $\lvert111\rangle$ sulla catena,
§5.1 del notebook $D=0$), non informativo per il confronto RBS/$W$.

In [ ]:
FAMIGLIE_PLOT = ["A: Ry-CNOT-Ry", "D: Ry-RBS-Ry", "RBS-1q.9",
                 "RBS-2q.8", "RBS-2qC.K2", "W-1q.9", "W-2q.8", "W-2qC.K2"]

def get_series(tag, name):
    ys = []
    for b in B_GRID:
        key = f"{tag}|{name}|{b}"
        ys.append(cache[key]["fidelity"] if key in cache else np.nan)
    return np.array(B_GRID), np.array(ys)

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True)
for ax, name in zip(axes.flat, FAMIGLIE_PLOT):
    b0, f0 = get_series("D0", name)
    bd, fd = get_series("DM", name)
    ax.plot(b0, f0, "o-", color="tab:blue", label="D=0")
    ax.plot(bd, fd, "s--", color="tab:red", label="DM")
    ax.axvline(BC, color="0.6", ls=":", lw=1)
    ax.set_title(name, fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("b")
axes[0, 0].set_ylabel("fidelity")
axes[1, 0].set_ylabel("fidelity")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Trimero a catena — fidelity vs b, per famiglia di ansatz")
fig.tight_layout()
plt.savefig("confronto_ansatz_trimero_catena_dm_grid.png", dpi=130)
plt.show()
print("[ok] figura salvata")

## 5. Lettura dei risultati

**A $D=0$**, tutte le famiglie con espressività sufficiente (6+ parametri o
2 giri) raggiungono $\mathcal F=1$ esatto — coerente con la Fase 4 (VQE senza
DM), qui riconfermato sulla griglia estesa.

**Con DM**, il quadro non è un mirror dell'anello: **né** RBS-2q **né** W-2q a
un solo giro raggiungono l'esatto (tetti $0.9626$ e $0.9937$ rispettivamente,
verificati con 60+ restart in `indagine_bc_trimero_catena.ipynb`) — sull'anello
bastava un solo giro di $W$. Servono due giri completi (`*-2qC.K2`) per
l'esatto su entrambe le famiglie. La gerarchia $W>$RBS sotto DM si conferma
comunque in termini relativi.

## 6. Cosa portarsi dietro

Candidato canonico per il seguito: **`W-2qC.K2`** (10 parametri, 8 gate a due
qubit, 19 gate totali contro i 39 di `RBS-2qC.K2` a parità di parametri e gate
a due qubit) — non l'analogo diretto di `W-2q.6` dell'anello, che qui sarebbe
insufficiente.